# vLLM 0.33 upgrade smoke test

Companion notebook for **"vLLM 0.33, Plainly: CUDA 13, Model Runner V2, and What Breaks on Upgrade"** (GDE series #1, Nov 9, 2026).

vLLM 0.33.0 shipped Nov 2, 2026: 927 commits, CUDA 13.0 only, PyTorch 2.14.0, Model Runner V2 as the only engine, and three deletions (Medusa, the MLP speculator, the legacy Ray compiled-graph executor).

This notebook runs anywhere — no GPU, no vLLM install needed. It checks the upgrade gates, reproduces the release's headline arithmetic (the 2.47x DSpark scheduler fix, the FLy throughput-vs-accuracy trade, the ABI-wheel rebuild tax), and prints the exact `vllm serve` + `curl` smoke-test commands to run on your GPU box.

## 1. Environment check — what is actually installed here?

In [ ]:
import importlib.util, shutil, subprocess

def _ver(mod):
    spec = importlib.util.find_spec(mod)
    if spec is None:
        return None
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return "installed (version unknown)"

print("vllm :", _ver("vllm"))
print("torch:", _ver("torch"))
print("nvidia-smi:", shutil.which("nvidia-smi") or "not found (no GPU driver visible)")
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi", "--query-gpu=driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
    print("driver:", out.stdout.strip() or "nvidia-smi query failed")
print()
print("This notebook's arithmetic does not need a GPU — the checks below are")
print("pure version-gate logic and the release's own reported numbers.")


## 2. Upgrade gates — the five things that decide whether 0.33 installs and serves

vLLM 0.33.0 requires: CUDA 13.0 artifacts (12.9 discontinued), `torch==2.14.0` on PyPI, NIXL ≥ 1.5 for the NIXL connector, and no references to Medusa / the MLP speculator / the legacy Ray executor. Paused generation now returns HTTP 503 instead of queueing — a client-behavior gate, not an install gate.

In [ ]:
def vtuple(s):
    return tuple(int(x) for x in str(s).split(".")[:3])

# (gate, installed, required, rule) — rule is a lambda(installed, required) -> bool
gates = [
    ("CUDA artifacts",      "13.0",  "13.0", lambda i, r: vtuple(i) >= vtuple(r)),
    ("CUDA artifacts (old)", "12.9", "13.0", lambda i, r: vtuple(i) >= vtuple(r)),
    ("torch (PyPI pin)",     "2.14.0", "2.14.0", lambda i, r: vtuple(i) == vtuple(r)),
    ("torch (newer, 2.15)",  "2.15.0", "2.14.0", lambda i, r: vtuple(i) == vtuple(r)),
    ("NIXL connector",       "1.5",   "1.5",  lambda i, r: vtuple(i) >= vtuple(r)),
    ("NIXL connector (old)", "1.4",   "1.5",  lambda i, r: vtuple(i) >= vtuple(r)),
]
for name, inst, req, rule in gates:
    print(f"{name:22s} installed={inst:7s} required={req:7s} -> {'PASS' if rule(inst, req) else 'FAIL'}")

# the gates must behave exactly as the release notes say
assert gates[0][3]("13.0", "13.0") is True    # CUDA 13.0 artifacts: OK
assert gates[1][3]("12.9", "13.0") is False   # CUDA 12.9: discontinued, fails
assert gates[2][3]("2.14.0", "2.14.0") is True
assert gates[3][3]("2.15.0", "2.14.0") is False  # PyPI pin is exact; 2.15 gets downgraded
assert gates[4][3]("1.5", "1.5") is True
assert gates[5][3]("1.4", "1.5") is False     # NIXL < 1.5 will not connect
print("\nAll six gates behave as documented: 12.9 fails, torch 2.15 fails the exact pin, NIXL 1.4 fails.")

REMOVED = ["medusa", "mlp_speculator", "ray_compiled_graph_executor"]
sample_config = {"speculative_config": {"model": "eagle"}, "executor": "uni"}
hits = [r for r in REMOVED if r in str(sample_config).lower()]
print("Removed-feature scan of a sample config:", hits or "clean — no removed features referenced")
assert hits == []

## 3. The DSpark scheduler fix — 1,373 → 3,390 tokens/s with no kernel change

The PR story: 2–32 leftover draft tokens kept splitting each new prefill into small chunks, so most decode steps mixed prefill into the batch and missed the full decode CUDA graph (ITL p50: 169 ms). The 0.33 scheduler stops drafters reserving token-budget slots they don't use, so the batch stays pure-decode. Toy simulation below: with leftover tokens present, most steps are mixed; without them, every step takes the fast path.

In [ ]:
import random
random.seed(0)

# --- headline arithmetic, from the cited PR (DeepSeek-V4.1-Flash + DSpark, GB300)
before_tps, after_tps = 1373, 3390
speedup = after_tps / before_tps
before_itl, after_itl = 169, 30   # ms, p50
itl_gain = before_itl / after_itl
print(f"decode throughput: {before_tps:,} -> {after_tps:,} tok/s  = {speedup:.2f}x")
print(f"ITL p50:           {before_itl} -> {after_itl} ms          = {itl_gain:.1f}x faster")
assert abs(speedup - 2.47) < 0.01
assert abs(itl_gain - 5.63) < 0.05

# --- toy: leftover tokens fragment the batch
# Each step, leftover draft tokens (2-32) force a small prefill chunk into the
# decode batch. A step is "pure decode" (CUDA-graph fast path) only when the
# leftover count is 0. Model: leftover ~ Uniform{2..32} before the fix, 0 after.
def mixed_step_fraction(n_steps=10000, fix=False):
    mixed = 0
    for _ in range(n_steps):
        leftover = 0 if fix else random.randint(2, 32)
        if leftover > 0:
            mixed += 1
    return mixed / n_steps

before = mixed_step_fraction(fix=False)
after = mixed_step_fraction(fix=True)
print(f"\nmixed (slow-path) steps: before fix {before:.1%}, after fix {after:.1%}")
assert before > 0.99 and after == 0.0
print("The fix removes the fragmentation entirely — that is where the 2.47x comes from.")

## 4. FLy — the lossy verifier, priced

FLy (opt-in, PR #53987) trades output-distribution fidelity for decode throughput: 1.041x–1.310x over standard speculative decoding across the PR's 48 model/platform/batch points (median 1.147x), with minimum accuracy retention of 96.3% vs target-only decoding. Standard verification stays the default and preserves the distribution exactly.

In [ ]:
# PR-reported FLy numbers (4 models x 6 batch sizes x 2 platforms = 48 points)
fly_min, fly_median, fly_max = 1.041, 1.147, 1.310
acc_retention_min = 96.3   # % vs target-only decoding
print(f"FLy throughput vs standard spec decode: {fly_min:.3f}x – {fly_max:.3f}x (median {fly_median:.3f}x)")
print(f"minimum accuracy retention: {acc_retention_min:.1f}%  (worst case: {100 - acc_retention_min:.1f} points lost)")
assert fly_min < fly_median < fly_max
assert 0 < acc_retention_min < 100

def fly_recommendation(accuracy_budget_pp, human_in_loop):
    """accuracy_budget_pp: points of accuracy you can afford to lose.
    Returns True only if the worst case fits the budget AND a human reviews output."""
    worst_loss = 100 - acc_retention_min
    return accuracy_budget_pp >= worst_loss and human_in_loop

cases = [
    ("free-form chat draft",      5.0, True,  True),
    ("code that ships unreviewed", 5.0, False, False),
    ("internal summary, human reads it", 4.0, True, True),
    ("data that trains the next model", 1.0, True, False),  # 3.7pp worst loss > 1pp budget
]
for name, budget, hil, expect in cases:
    got = fly_recommendation(budget, hil)
    print(f"{name:34s} budget={budget}pp human={hil!s:5s} -> {'FLy OK' if got else 'keep standard'}")
    assert got == expect
print("\nRule: FLy only where a different token is harmless AND a human is in the loop.")

## 5. The ABI-stable wheel — the rebuild tax, and the smoke test

Across the last four torch releases vLLM took 33, 34, 44, and 60 days to ship the first release pinning each one (torch 2.12 was never pinned). The 0.33 wheel is ABI-stable with a minimum torch of 2.13.0 — but PyPI metadata still pins `torch==2.14.0`, so pip won't install it against newer torch yet. The cell below quantifies the tax and prints the smoke-test commands for your GPU box, pinned to 0.33.0.

In [ ]:
days_to_pin = {"torch 2.10": 33, "torch 2.11": 34, "torch 2.13": 44, "torch 2.14": 60}
mean_tax = sum(days_to_pin.values()) / len(days_to_pin)
print("days from torch release to first vLLM release pinning it:")
for k, v in days_to_pin.items():
    print(f"  {k}: {v} days")
print(f"mean rebuild tax: {mean_tax:.1f} days  (torch 2.12: never pinned)")
assert abs(mean_tax - 42.75) < 0.01
assert max(days_to_pin.values()) == 60

# ABI-stable wheel audit (independent audit of the published 336 MB wheel)
min_torch_stable = "2.13.0"
print(f"\n0.33 wheel: Torch ABI STABLE, min torch {min_torch_stable}, 13 extensions, 0 unstable symbols")
print("PyPI metadata still pins torch==2.14.0 — binary capability ahead of packaging by one release.")

# --- smoke test commands, pinned to 0.33.0 (run on your GPU box, not here) ---
MODEL = "Qwen/Qwen3-0.6B"   # small model: boots fast, proves the server works
cmds = f'''# 1. fresh env + install (CUDA 13 driver already updated)
pip install "vllm==0.33.0"

# 2. serve a small model (Model Runner V2 is the engine; no flags needed)
vllm serve {MODEL} --port 8000

# 3. smoke test — in a second shell
curl http://localhost:8000/v1/models | head -c 300; echo
curl http://localhost:8000/v1/completions -H "Content-Type: application/json" -d '{{"model": "{MODEL}", "prompt": "The capital of France is", "max_tokens": 16}}'
'''
print("\n" + cmds)
assert "vllm==0.33.0" in cmds and "vllm serve" in cmds and "/v1/completions" in cmds
print("Smoke test ready: expect a 200 from /v1/models and a completion ending in 'Paris'.")